# Databricks notebook source
## 00 — Raw → Bronze : Référentiel Communes IDF (IGN)

In [0]:
%run ../utils/init.py

In [0]:
%pip install geopandas
import geopandas as gpd

local_path = "/tmp/IDF_commune.geojson"
dbutils.fs.cp(f"{RAW}/ign/IDF_commune.geojson", f"file:{local_path}")

gdf = gpd.read_file(local_path)
print(f"CRS source : {gdf.crs}")
print(f"Lignes : {len(gdf)}")
print(gdf.columns.tolist())

In [0]:
# Reprojection 2154 → 4326
gdf = gdf.to_crs(epsg=4326)

# Colonnes dérivées
gdf["centroid_lon"] = gdf.geometry.centroid.x
gdf["centroid_lat"] = gdf.geometry.centroid.y
gdf["surface_km2"]  = gdf.geometry.to_crs(epsg=2154).area / 1_000_000
gdf["geometry_wkt"] = gdf.geometry.apply(lambda g: g.wkt)

# Sélection des colonnes utiles
gdf_clean = gdf[[
    "code_insee",
    "nom_officiel",
    "code_insee_du_departement",
    "population",
    "centroid_lon",
    "centroid_lat",
    "surface_km2",
    "geometry_wkt"
]].rename(columns={
    "code_insee": "code_commune",
    "nom_officiel": "nom_commune",
    "code_insee_du_departement": "code_dept"
})

# Conversion Spark
df = spark.createDataFrame(gdf_clean)
print(f"✅ {df.count()} communes converties")
df.show(3)

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{BRONZE}/referentiel_communes/")

print("✅ Table bronze/referentiel_communes écrite")